# Climat Méditerranée — Tunisie & Europe (1980–2024)
## Migration d'un pipeline R vers Python sur données ERA5 (Copernicus)

**Cours :** ANALYSE DES DONNÉES — ESSAI · Mme Tej
**Étudiant :** Bacem Ben Ahmad

---

## Question scientifique

> **Comment les vagues de chaleur et le stress hydrique ont-ils évolué entre 1980 et 2024 sur le bassin méditerranéen ? La rive sud (Tunisie) et la rive nord (Europe méridionale) se réchauffent-elles au même rythme, et quelles sont les relations entre température, humidité et précipitations ?**

### Justification
1. **Pertinence locale et comparative** — Tunisie (rive sud) et Europe méridionale (rive nord) bordent le même bassin, identifié par le GIEC comme un *climate change hotspot*. Comparer les deux rives permet de tester si le réchauffement est homogène ou asymétrique.
2. **Dimensions multiples** — 3 variables climatiques (T°, humidité, précipitations) + 1 dérivée → étude de **relations** physiques.
3. **Période 1980–2024** — 4 décennies, suffisant pour distinguer une tendance climatique du bruit interannuel.
4. **Vérifiabilité** — coefficients de tendance °C/décennie, nombre de jours > 30 °C, mm cumulés saisonniers.

## Source des données

**Climate Data Store (CDS) — Copernicus / ECMWF** · https://cds.climate.copernicus.eu

| | |
|---|---|
| Dataset | `reanalysis-era5-single-levels` |
| Type | Réanalyse (observations + modèle physique) |
| Résolution spatiale | 0.25° × 0.25° (~28 km) |
| Résolution temporelle | 1 valeur quotidienne à 12 UTC |
| Format | NetCDF, lu via `xarray` |

### Variables (3 + 1 dérivée)

| Variable CDS | Symbole | Unité | Rôle |
|---|---|---|---|
| `2m_temperature` | `t2m` | °C | Vagues de chaleur |
| `2m_dewpoint_temperature` | `d2m` | °C | Calcul humidité relative |
| `total_precipitation` | `tp` | mm | Stress hydrique |
| **dérivée** Magnus-Tetens | `rh` | % | Humidité relative |

### Zones d'analyse

| Zone | Bornes | Description |
|---|---|---|
| **Tunisie** (rive sud) | 30–37.5°N / 7.5–12°E | Climat semi-aride méditerranéen |
| **Europe** (rive nord) | 40–46°N / -10–40°E | Espagne, sud France, Italie, Balkans, Grèce |
| **Méditerranée** (bassin entier) | 28–46°N / -10–40°E | Référence régionale |

## Migration R → Python

| # | R (S1) | Python (S2) |
|---|---|---|
| 1 | `read.csv()` / `read_nc()` | `xr.open_dataset()` |
| 2 | `dplyr::filter()` | `df.query()` / boolean indexing |
| 3 | `dplyr::group_by()` + `summarise()` | `df.groupby().agg()` |
| 4 | `tidyr::pivot_longer/wider` | `df.melt()` / `df.pivot()` |
| 5 | `mutate(... = ...)` | xarray broadcasting / `df.assign()` |
| 6 | `is.na()`, `na.omit()` | `df.isna()`, `df.dropna()` |
| 7 | `boxplot.stats()` (IQR) | `df.quantile()`, `scipy.stats.zscore` |
| 8 | `cor()` | `df.corr()` |
| 9 | `lm()` | `scipy.stats.linregress`, `numpy.polyfit` |
| 10 | `Kendall::MannKendall()` | `scipy.stats.kendalltau` |
| 11 | `ggplot + geom_*` | `matplotlib`, `seaborn` |
| 12 | `shinydashboard` | **Dash + Plotly** (imposé) |

**Difficultés techniques** : passage du tabulaire R au multi-dim xarray (time × lat × lon), ZIP CDS contenant 2 NetCDF (instantané + accumulé) à fusionner via `xr.merge`, dépendance `dask` pour `open_mfdataset`.

---
# Partie 1 — Données & qualité

## 1.1 Configuration et chargement

In [ ]:
# Imports + config globale (une seule fois pour tout le notebook)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110,
    'axes.titleweight': 'bold', 'axes.titlesize': 12,
    'font.size': 11,
})

# --- Constantes du projet ------------------------------------------------
ZONE_COLORS = {'europe': '#3CAEA3',  'tunisie': '#F6993F', 'mediterranee': '#0F4C81'}
ZONE_LABELS = {'europe': 'Europe (rive nord)',
               'tunisie': 'Tunisie (rive sud)',
               'mediterranee': 'Méditerranée (bassin)'}
ZONES_ORDER = ['europe', 'tunisie', 'mediterranee']
REF_PERIOD = (1981, 2010)  # référence climatologique standard WMO

# --- Chargement ----------------------------------------------------------
DATA = Path('..').resolve() / 'data' / 'processed' / 'era5_daily.parquet'
df = pd.read_parquet(DATA)
df['date']   = pd.to_datetime(df['date'])
df['decade'] = (df['year'] // 10) * 10

print(f'Lignes  : {len(df):,}')
print(f'Période : {df.date.min().date()} → {df.date.max().date()}')
print(f'Zones   : {df.zone.unique().tolist()}')
df.head()

**Interprétation.** 49 311 observations quotidiennes (16 437 jours × 3 zones) sur 45 ans. Chaque ligne = moyenne spatiale d'une variable climatique sur une zone à 12 UTC.

## 1.2 Audit qualité — valeurs manquantes & aberrantes

In [ ]:
VARS = ['t2m', 'd2m', 'rh', 'tp']

# 1. Taux de NA — vectorisé sur toutes les colonnes en une passe
na_pct = (df.groupby('zone')[VARS]
            .apply(lambda g: g.isna().mean() * 100)
            .round(3))
print('Taux de NA (%) :')
print(na_pct.to_string())

# 2. Outliers — IQR + z-score, en une seule passe vectorisée
def outlier_summary(s):
    s = s.dropna().to_numpy()
    n = len(s)
    q1, q3 = np.quantile(s, [.25, .75])
    iqr = q3 - q1
    n_iqr = int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum())
    n_z   = int((np.abs(stats.zscore(s)) > 3).sum())
    return pd.Series({'IQR_outliers': n_iqr, 'pct_IQR': round(100*n_iqr/n, 2),
                      'z>3': n_z, 'pct_z>3': round(100*n_z/n, 2)})

outliers = df[VARS].apply(outlier_summary).T
print('\nOutliers globaux :')
print(outliers.to_string())

**Interprétation.**
- **Aucun NA** : ERA5 est une réanalyse complète (le modèle remplit chaque cellule de la grille à chaque pas de temps). Cela contraste avec un dataset d'observations directes.
- **Outliers conservés** : pour les précipitations notamment (~2,5 % d'outliers IQR), il s'agit de jours d'orage ou de pluies diluviennes — les supprimer biaiserait l'étude des extrêmes. Décision méthodologique : on les **identifie**, on ne les **supprime pas**.

---
# Partie 2 — Signal climatique : tendances, anomalies, extrêmes

## 2.1 Tendance long terme (Mann-Kendall + pente de Sen)

Pour chaque zone, on calcule la pente de tendance par 3 méthodes :
- **OLS** (régression linéaire paramétrique) — sensible aux outliers
- **Pente de Sen** — médiane de toutes les pentes, robuste aux outliers
- **Test de Mann-Kendall** — test non-paramétrique de tendance, **standard OMM**

In [ ]:
def sen_slope(x):
    """Pente de Sen : médiane de toutes les pentes inter-points (vectorisé)."""
    x = np.asarray(x, dtype=float)
    i, j = np.triu_indices(len(x), k=1)
    return float(np.median((x[j] - x[i]) / (j - i)))

def trend_stats(y, x=None):
    """Renvoie OLS, Sen, Kendall τ, p-MK en une seule passe."""
    y = np.asarray(y, dtype=float)
    x = np.arange(len(y)) if x is None else np.asarray(x, dtype=float)
    slope, intercept, r, _, _ = stats.linregress(x, y)
    tau, p_mk = stats.kendalltau(x, y)
    return dict(slope=slope, intercept=intercept, r=r,
                sen=sen_slope(y), tau=tau, p_mk=p_mk)

annual = df.groupby(['year','zone']).t2m.mean().unstack('zone')

fig, ax = plt.subplots(figsize=(12, 5))
rows = []
for zone in ZONES_ORDER:
    s = annual[zone]; color = ZONE_COLORS[zone]
    t = trend_stats(s.values, s.index.values)
    ax.plot(s.index, s.values, lw=2.2, color=color, label=ZONE_LABELS[zone])
    ax.plot(s.index, t['slope']*s.index + t['intercept'],
            '--', color=color, alpha=0.6, lw=1.4)
    rows.append({'zone': zone, 'OLS_°C/déc': t['slope']*10,
                 'Sen_°C/déc': t['sen']*10, 'τ_Kendall': t['tau'],
                 'p_MK': t['p_mk']})

ax.set_title('Température moyenne annuelle à 12 UTC — 3 zones (1980–2024)')
ax.set_xlabel('Année'); ax.set_ylabel('T 2m (°C)')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('\nTests statistiques de tendance :')
print(pd.DataFrame(rows).set_index('zone').round(4).to_string())

**Interprétation.**
- Les **trois zones** se réchauffent significativement (p < 0,001 sur Mann-Kendall pour chacune).
- **Europe : +0,51 °C/décennie**, **Tunisie : +0,45 °C/décennie**, **Méditerranée : +0,46 °C/décennie**.
- La rive nord se réchauffe **légèrement plus vite** — effet d'amplification continentale (terres européennes vs mer qui amortit la rive sud).
- Ce rythme est environ **2× la moyenne mondiale** (~0,2 °C/déc selon GIEC) → le bassin est bien un *climate change hotspot*.
- L'accord OLS / Sen confirme l'absence d'outliers influents — la tendance est robuste.

## 2.2 Anomalies vs climatologie 1981–2010 (référence WMO)

L'**anomalie** est l'écart entre la valeur d'une année et la moyenne d'une période de référence standardisée par l'OMM (1981–2010). Elle élimine les différences de climat de base entre zones et révèle uniquement le **signal de tendance**.

In [ ]:
ref = annual.loc[REF_PERIOD[0]:REF_PERIOD[1]].mean()
anomalies = annual.sub(ref)  # broadcasting vectorisé

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for ax, zone in zip(axes, ZONES_ORDER):
    a = anomalies[zone].to_numpy()
    years = anomalies.index.to_numpy()
    ax.bar(years, a, color=np.where(a > 0, '#DC2626', '#3B82F6'),
           edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.axvspan(REF_PERIOD[0]-0.5, REF_PERIOD[1]+0.5, color='gray', alpha=0.10)
    ax.set_title(f'{ZONE_LABELS[zone]} — réf. = {ref[zone]:.2f} °C', loc='left')
    ax.set_ylabel('Anomalie (°C)'); ax.grid(alpha=0.3, axis='y')
axes[-1].set_xlabel('Année')
fig.suptitle('Anomalies de température vs climatologie 1981–2010 (référence WMO)',
             fontweight='bold', fontsize=13, y=1.00)
plt.tight_layout(); plt.show()

dec_anom = anomalies.groupby(anomalies.index // 10 * 10).mean().round(2)
dec_anom.index.name = 'decade'
print('Anomalies moyennes par décennie (°C) :')
print(dec_anom.to_string())

**Interprétation.** Le pattern est sans ambiguïté :
- Avant 1995 : barres bleues (sous la normale) dominent.
- Après 2000 : la quasi-totalité des barres sont **rouges**, et leur magnitude croît.
- 2015–2024 : anomalies de **+1 à +2 °C** sur les 3 zones — considérable sur une moyenne annuelle.

Cette représentation est **plus parlante** que les valeurs brutes : elle isole le signal du niveau climatique de base.

## 2.3 Évolution des canicules (jours T midi > 30 °C en été)

In [ ]:
hot_summer = (df.loc[df.season == 'JJA']
                .assign(hot=df.t2m > 30)
                .groupby(['year','zone']).hot.sum()
                .unstack('zone'))
rolling5 = hot_summer.rolling(5, center=True).mean()  # vectorisé sur les 3 zones

fig, ax = plt.subplots(figsize=(13, 5))
for zone in ZONES_ORDER:
    color = ZONE_COLORS[zone]
    ax.plot(hot_summer.index, hot_summer[zone], marker='o', ms=4, lw=1,
            color=color, alpha=0.45, label=f'{ZONE_LABELS[zone]} — brut')
    ax.plot(rolling5.index, rolling5[zone], lw=3, color=color,
            label=f'{ZONE_LABELS[zone]} — moy. mob. 5 ans')
ax.set_title("Nombre de jours d'été (JJA) avec T midi > 30 °C")
ax.set_xlabel('Année'); ax.set_ylabel('Nb jours')
ax.legend(ncol=2, fontsize=9, loc='upper left'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Comparaison périodique — np.where + groupby (plus rapide que pd.cut)
periode = np.where(hot_summer.index <= 2000, '1980-2000', '2001-2024')
summary = hot_summer.groupby(periode).mean().round(1).T
summary['Δ jours'] = (summary['2001-2024'] - summary['1980-2000']).round(1)
summary['Δ %']    = (100 * summary['Δ jours'] / summary['1980-2000']).round(0)
print('Jours T>30°C par été — moyenne par période :')
print(summary.to_string())

**Interprétation.**
- **Tunisie** : ~70 jours/an (1980–2000) → ~90 jours/an (2001–2024), avec des pics à 100+ jours sur les années récentes.
- **Europe méridionale** : passage de quasi-nul à plusieurs dizaines de jours par an — **émergence** d'un phénomène nouveau.
- **Une moyenne qui monte de 1–2 °C déplace toute la distribution** : les extrêmes chauds gagnent beaucoup plus en probabilité que ne le suggère le décalage moyen.

---
# Partie 3 — Structure climatique & relations entre variables

## 3.1 Cycle saisonnier comparé (3 zones, 3 variables)

In [ ]:
months_fr = ['Jan','Fév','Mar','Avr','Mai','Juin','Juil','Aoû','Sep','Oct','Nov','Déc']

# Pivot : index = mois, colonnes = (variable, zone) — une seule passe
monthly = (df.groupby(['month','zone'])[['t2m','rh','tp']]
             .mean().unstack('zone'))

VAR_LABELS = {'t2m': 'Température (°C)',
              'rh':  'Humidité relative (%)',
              'tp':  'Précipitations moy. (mm/h à 12 UTC)'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, var in zip(axes, VAR_LABELS):
    for zone in ZONES_ORDER:
        ax.plot(monthly.index, monthly[(var, zone)],
                marker='o', ms=8, lw=2.5,
                color=ZONE_COLORS[zone], label=ZONE_LABELS[zone])
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(months_fr)
    ax.set_title(VAR_LABELS[var]); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle('Cycle saisonnier moyen par zone (1980–2024)',
             fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

**Interprétation.**
- **Amplitude T°** : Tunisie oscille entre ~13 °C (janvier) et ~33 °C (juillet) ; Europe reste plus modérée. La Méditerranée (qui inclut la mer) lisse le tout.
- **Humidité** : minimum estival commun aux 3 zones, maximum hivernal — typique du climat méditerranéen.
- **Précipitations** : maximum **hivernal** en Tunisie et Méditerranée (climat méditerranéen classique), tandis que l'Europe méridionale a un cycle plus plat.
- L'**anti-corrélation T°/RH** est visible directement : les courbes T° et RH évoluent en miroir.

## 3.2 Précipitations & stress hydrique (Tunisie)

In [ ]:
SEASON_ORDER  = ['DJF','MAM','JJA','SON']
SEASON_COLORS = {'DJF':'#3B82F6', 'MAM':'#10B981',
                 'JJA':'#F59E0B', 'SON':'#EF4444'}

rain_seasons = (df.loc[df.zone == 'tunisie']
                  .groupby(['year','season']).tp.sum()
                  .unstack('season').reindex(columns=SEASON_ORDER))

fig, ax = plt.subplots(figsize=(13, 5))
for season in SEASON_ORDER:
    ax.plot(rain_seasons.index, rain_seasons[season],
            marker='o', ms=4, alpha=0.85,
            color=SEASON_COLORS[season], label=season)
ax.set_title('Précipitations cumulées par saison — Tunisie')
ax.set_xlabel('Année'); ax.set_ylabel('mm cumulés sur la saison')
ax.legend(title='Saison'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Tendances saisonnières — toutes en une seule apply
trends_seasons = rain_seasons.apply(
    lambda s: pd.Series(stats.linregress(s.dropna().index,
                                         s.dropna().values)._asdict()))
trends_seasons.loc['pente_mm/déc'] = trends_seasons.loc['slope'] * 10
print('Tendances saisonnières (Tunisie) :')
for season in SEASON_ORDER:
    s = trends_seasons[season]
    sig = '*' if s['pvalue'] < 0.05 else ' '
    print(f'  {season}: pente = {s["pente_mm/déc"]:+6.2f} mm/déc  '
          f'(r={s["rvalue"]:+.2f}, p={s["pvalue"]:.3f}) {sig}')

**Interprétation.**
- **DJF** (hiver) est le principal contributeur des précipitations annuelles ; **JJA** est quasi-nul (climat méditerranéen).
- Aucune tendance significative à p < 0,05 sur 45 ans — typique : la **variabilité interannuelle** des précipitations méditerranéennes est forte et masque le signal long terme.
- Le **stress hydrique** se manifeste donc moins par une chute des précipitations que par la **combinaison T° ↑ et RH ↓** : à pluviométrie constante, l'évapotranspiration accrue dégrade le bilan hydrique.

## 3.3 Corrélations T° / RH / Précipitations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, zone in zip(axes, ZONES_ORDER):
    corr = df[df.zone == zone][['t2m','d2m','rh','tp']].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, square=True,
                ax=ax, cbar_kws={'shrink': 0.7})
    ax.set_title(ZONE_LABELS[zone])
fig.suptitle('Matrices de corrélation par zone',
             fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

**Interprétation.** Trois relations physiques se confirment :
1. **t2m / rh très anti-corrélées** (≈ -0,7 sur les 3 zones) : air chaud → capacité de vapeur ↑ → RH ↓ à humidité absolue constante (Clausius-Clapeyron).
2. **t2m / d2m positivement corrélées** (≈ +0,7) : les masses d'air chaud sont aussi plus humides en absolu.
3. **tp** faiblement corrélée — les précipitations sont des événements ponctuels qui ne suivent pas la T° en continu.

**Implication** : les vagues de chaleur en Tunisie sont accompagnées d'une chute de l'humidité, ce qui aggrave le stress hydrique sur l'agriculture et les écosystèmes.

---
# Partie 4 — Visualisations iconiques du changement climatique

Deux représentations qui sortent des plots standards : utilisées dans la **communication scientifique internationale** (Met Office, BBC, NYT). Elles synthétisent visuellement le signal climatique mieux que n'importe quelle série temporelle classique.

## 4.1 Warming Stripes (Ed Hawkins, 2018)

Inventées par le climatologue **Ed Hawkins** (University of Reading), les *warming stripes* sont devenues le **symbole graphique mondial** du réchauffement climatique. Chaque bande verticale = une année, colorée selon l'écart à la moyenne 1981–2010. Pas d'axes, pas de chiffres : **juste le signal climatique brut**.

In [ ]:
# Z-score vs période de référence (broadcasting vectorisé)
ref_window = annual.loc[REF_PERIOD[0]:REF_PERIOD[1]]
stripes = (annual - ref_window.mean()) / ref_window.std()

# Palette officielle Hawkins (bleu profond -> rouge profond)
hawkins_cmap = mcolors.LinearSegmentedColormap.from_list('hawkins', [
    '#08306B', '#2171B5', '#6BAED6', '#C6DBEF',
    '#FEE0D2', '#FB6A4A', '#CB181D', '#67000D'])
norm = mcolors.Normalize(vmin=-2.5, vmax=2.5)

fig, axes = plt.subplots(3, 1, figsize=(14, 5))
years = stripes.index.to_numpy()
for ax, zone in zip(axes, ZONES_ORDER):
    # imshow vectorisé : 1 ligne, N années — bien plus rapide que axvspan
    z = stripes[zone].to_numpy().reshape(1, -1)
    ax.imshow(z, aspect='auto', cmap=hawkins_cmap, norm=norm,
              extent=(years.min()-0.5, years.max()+0.5, 0, 1))
    ax.set_yticks([])
    ax.set_xticks([1980, 1990, 2000, 2010, 2020, 2024])
    ax.set_title(f'{ZONE_LABELS[zone]} — Warming Stripes 1980–2024',
                 loc='left', fontsize=11)
fig.suptitle('Warming Stripes — anomalies vs 1981–2010 (style Ed Hawkins)',
             fontweight='bold', fontsize=14, y=1.00)
plt.tight_layout(); plt.show()

**Interprétation.** Sans aucun chiffre, on voit :
- Les **années 1980–1990** dominent en bleu (sous la normale).
- À partir des **années 2000**, le rouge envahit progressivement l'image.
- Les **5–10 dernières années** sont rouge profond presque partout.

Cette image traverse les médias depuis 2018 — c'est une "carte d'identité visuelle" du réchauffement, immédiatement compréhensible par n'importe quel public.

## 4.2 Phénologie de l'été — Tunisie

Plutôt que de **compter** les jours chauds, on regarde **quand** ils arrivent. Premier jour > 30 °C, dernier jour > 30 °C, et la fenêtre entre les deux.

In [ ]:
# Filtrage + agrégation min/max en une seule passe
hot_tun = (df.loc[(df.zone == 'tunisie') & (df.t2m > 30)]
             .assign(doy=lambda d: d.date.dt.dayofyear)
             .groupby('year').doy.agg(['min', 'max'])
             .rename(columns={'min': 'first', 'max': 'last'}))
hot_tun['span'] = hot_tun['last'] - hot_tun['first']

# Régressions linéaires (récupère slope + intercept en une passe)
years = hot_tun.index.to_numpy()
reg_first = stats.linregress(years, hot_tun['first'])
reg_last  = stats.linregress(years, hot_tun['last'])
trend_first = reg_first.slope * years + reg_first.intercept
trend_last  = reg_last.slope  * years + reg_last.intercept

fig, ax = plt.subplots(figsize=(14, 6.5))
ax.barh(years, hot_tun['span'], left=hot_tun['first'],
        height=0.75, color='#F59E0B', alpha=0.55, edgecolor='none')
ax.scatter(hot_tun['first'], years, color='#10B981', s=45, zorder=5,
           label='Premier jour > 30 °C', edgecolor='white')
ax.scatter(hot_tun['last'],  years, color='#DC2626', s=45, zorder=5,
           label='Dernier jour > 30 °C', edgecolor='white')
ax.plot(trend_first, years, color='#10B981', ls='--', lw=2,
        label=f'Tendance début : {reg_first.slope*10:+.1f} j/déc')
ax.plot(trend_last,  years, color='#DC2626', ls='--', lw=2,
        label=f'Tendance fin : {reg_last.slope*10:+.1f} j/déc')

month_starts = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
ax.set_xticks(month_starts); ax.set_xticklabels(months_fr)
ax.set_xlim(0, 366); ax.invert_yaxis()
ax.set_ylabel('Année')
ax.set_title("Phénologie de l'été en Tunisie — fenêtre des jours > 30 °C")
ax.legend(loc='lower right', framealpha=0.95); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

# Comparaison périodique vectorisée
periode = np.where(hot_tun.index <= 1989, '1980s',
           np.where(hot_tun.index >= 2015, '2015-2024', 'autre'))
window_summary = hot_tun.groupby(periode)[['first','last','span']].mean().round(0)
print('\nFenêtre saisonnière des jours > 30 °C en Tunisie :')
print(window_summary.loc[['1980s', '2015-2024']].to_string())

**Interprétation.** La fenêtre des jours > 30 °C **s'allonge** progressivement : le début glisse vers le printemps, la fin vers l'automne. La saison chaude tunisienne grignote sur les saisons intermédiaires.

Cette représentation **phénologique** quantifie le **calendrier biologique** plutôt que la chaleur seule : c'est le calendrier des cultures, du tourisme, et de la consommation d'eau d'irrigation.

---
# Partie 5 — Synthèse & conclusion

## 5.1 Tableau de synthèse — indicateurs clés (3 zones)

In [ ]:
# Tableau de synthèse — entièrement vectorisé (1 seul concat final)
VARS = ['t2m', 'rh', 'tp']

# 1. Moyennes par zone
means = df.groupby('zone')[VARS].mean().round(2).add_suffix('_moy')

# 2. Tendances (slope OLS par décennie) sur les moyennes annuelles
def slope_decade(s):
    s = s.dropna()
    return stats.linregress(s.index, s.values).slope * 10 if len(s) >= 3 else np.nan

trends = (df.groupby(['year','zone'])[VARS].mean()
            .unstack('zone').apply(slope_decade)
            .unstack().T.round(3)
            .add_prefix('tend_').add_suffix('/déc'))

# 3. Jours T > 30 °C / an, moyenne par zone
hot_days = (df.assign(hot=df.t2m > 30)
              .groupby(['year','zone']).hot.sum()
              .groupby('zone').mean().round(0)
              .rename('jours_T>30/an'))

# 4. Corrélation T/RH — extraction directe via xs (plus rapide que apply)
corr_t_rh = (df.groupby('zone')[['t2m','rh']].corr()
               .xs('t2m', level=1)['rh']
               .round(2).rename('corr_T_RH'))

synthese = pd.concat([means, trends, hot_days, corr_t_rh],
                     axis=1).loc[ZONES_ORDER]
print('=' * 85)
print('SYNTHÈSE — Europe / Tunisie / Méditerranée 1980–2024')
print('=' * 85)
print(synthese.to_string())

## 5.2 Conclusion — réponse à la question scientifique

> *Comment les vagues de chaleur et le stress hydrique ont-ils évolué entre 1980 et 2024 ? La rive sud (Tunisie) et la rive nord (Europe méridionale) se réchauffent-elles au même rythme ?*

Les données ERA5 montrent un **réchauffement net et statistiquement significatif** sur tout le bassin :

1. **Réchauffement quasi-uniforme** — Tunisie +0,45 °C/déc, Europe méridionale +0,51 °C/déc, Méditerranée +0,46 °C/déc. Tests Mann-Kendall **très significatifs** (p < 0,001) sur les 3 zones.
2. **L'Europe se réchauffe légèrement plus vite** que la Tunisie (effet d'amplification continentale).
3. **Multiplication des jours chauds** en Tunisie : ~70 → ~90 jours/an, +50 % sur 45 ans.
4. **Anomalies vs 1981–2010** : les 10 dernières années sont systématiquement +1 à +2 °C au-dessus de la normale WMO.
5. **Phénologie** : la fenêtre des jours > 30 °C en Tunisie **s'allonge** par les deux extrémités, début comme fin de saison.
6. **Anti-corrélation T°/RH** robuste (-0,7) sur les 3 zones — Clausius-Clapeyron confirmée.
7. **Précipitations** : pas de tendance significative à 45 ans, mais l'effet thermique seul suffit à expliquer un **stress hydrique accru** via l'évapotranspiration.

**Asymétrie nord/sud** : le rythme de réchauffement est légèrement plus fort au nord, mais c'est **la rive sud qui en subit les conséquences les plus dommageables** parce qu'elle part d'un climat déjà aride.

### Regard critique sur les données
- ERA5 = **réanalyse**, pas observation directe → assimilation observations + modèle. Densité de stations faible au Maghreb → incertitude plus élevée que sur l'Europe occidentale (Hersbach et al., 2020).
- **Résolution 0.25° (~28 km)** → microclimats côtiers / urbains lissés.
- **1 valeur quotidienne à 12 UTC** → cycle diurne perdu. Acceptable pour les tendances long terme, insuffisant pour les pics instantanés.
- **`tp` = accumulation horaire à 12 UTC**, pas le cumul quotidien.
- **Zone "Europe" limitée à 40–46°N** : couvre la rive nord méditerranéenne, pas l'Europe au-delà des Alpes.

### Méthodologie statistique
- **OLS** (paramétrique) + **Mann-Kendall** + **pente de Sen** (non-paramétriques, robustes) — standard OMM.
- **Anomalies** vs période de référence WMO 1981–2010.
- **Visualisations iconiques** : Warming Stripes (Hawkins 2018) + analyse phénologique (allongement de la saison chaude).
- **Module de prédiction séparé** (`projet_prediction.ipynb`) — comparaison naïf saisonnier / OLS / SARIMA / Random Forest sur la série mensuelle T2m de la Tunisie, validation 2015–2024 et projection 2025–2030.

### Perspectives
- Étendre la zone vers l'**Europe continentale** (jusqu'à 60°N).
- Affiner avec **ERA5-Land (0.1°)** sur la partie continentale.
- Croiser avec données **stations sol INM/Météo-France**.
- Analyser l'**humidité du sol** ERA5-Land (stress hydrique direct).
- Comparer aux **projections CMIP6** (scénarios SSP) pour le futur.